## 웹서비스 흐름
1. 클라이언트가 `http://example.com` 같은 URL로 요청을 보낸다.
2. 웹서버가 HTML 문서를 응답한다.
3. 웹브라우저는 HTML을 파싱해서 화면에 렌더링한다.

### 정적 웹페이지 스크래핑 전 확인
1. 대상 사이트의 이용약관을 확인한다.
2. `robots.txt`를 확인하여 크롤러 접근 정책을 확인한다.
3. `robots.txt`는 크롤러에게 “이 경로는 수집하지 말아 달라”고 알려주는 표준 정책 파일이다.
4. 단, `robots.txt`가 허용한다고 해서 저작권, 개인정보, 이용약관 문제가 모두 해결되는 것은 아니다.
5. 짧은 시간에 반복 요청을 보내면 서버에 부담을 줄 수 있으므로 요청 간격과 수집 범위를 조절한다.

### 정적 웹페이지란?
- 서버가 응답한 HTML 안에 필요한 데이터가 이미 포함된 페이지이다.
- `requests`로 HTML을 가져온 뒤 `BeautifulSoup`으로 원하는 태그를 찾을 수 있다.
- JavaScript 실행 후에 데이터가 생기는 페이지는 정적 방식만으로 수집이 어려울 수 있다.

In [3]:
from urllib import response
# requests, beautifulsoup4 모듈 설치
!pip install requests beautifulsoup4

## 웹 페이지 요청 및 파싱

In [8]:
import requests
from bs4 import BeautifulSoup

url = 'http://www.naver.com'

# 일부 사이트는 브라우저가 아닌 요청을 제한할 수 있으므로 User-Agent를 함께 전달한다.
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/125.0 Safari/537.36"
}

try :
    response = requests.get(url, headers=headers, timeout=10)

    # 에러 코드 400~ , 500~ 응답 코드 확인시 에러 발생
    response.raise_for_status()

except requests.exceptions.Timeout as err:
    print('시간 초과 :', err)

except requests.exceptions.RequestException as err:
    # 요청 오류
    print('요청 실패 :',err)

else: # 정상 응답
    html = response.text
    # print(html)

    # BeautifulSoup 객체 이용하여 파싱
    soup = BeautifulSoup(html, 'html.parser')
    print(soup.title)


<title>NAVER</title>


## find, find_all

In [22]:
from bs4 import BeautifulSoup

with open('sample.html', 'r', encoding='utf-8') as f:
    html = f.read() # html 문자열로 저장
    soup = BeautifulSoup(html, 'html.parser') # 파실

    print(soup.title)

    # find_all 모든 일치하는 태그 조회

    li_tags = soup.find_all('li')

    # print(li_tags)

    # for li_tag in li_tags:
        # print(li_tag, type(li_tag))
        # print(li_tag.text)

    # find : 일치 하는 첫 번쨰 태그 조회
    # find('태그명', {속성명 : 속성값})
    # find_all('태그명', {속성명 : 속성값})
    first_li = soup.find('li')
    print('first_li :',first_li.text)

    ht1_tag = soup.find('h1', {'id' : 'page-title'})
    print('ht1_tag :',ht1_tag)

    section_content_tags = soup.find_all('section', {'class' : 'section-content'})
    for section_content_tag in section_content_tags:
        print('section_content_tag.txt :',section_content_tag.txt) # 내용
        print('section_content_tag.attrs :',section_content_tag.attrs) # 속성

<title>샘플 HTML Page</title>
first_li : Section 1
ht1_tag : <h1 id="page-title">Welcome to the Sample HTML Page</h1>
section_content_tag.txt : None
section_content_tag.attrs : {'id': 'section1', 'class': ['section-content']}
section_content_tag.txt : None
section_content_tag.attrs : {'id': 'section2', 'class': ['section-content']}
section_content_tag.txt : None
section_content_tag.attrs : {'id': 'section3', 'class': ['section-content']}


## select, select_one
-CSS 선택자(태그명, #, ., >, ... 등)을 이용하여 조회

In [36]:
li_tags = soup.select('li')

# print(li_tags)

first_li = soup.select_one('li')

# print(first_li)

#id(#)을 이용해서 찾기
h1_tag = soup.select_one('#page-title')
# print('h1_tag :',h1_tag.text)

# class(.) 이용해서 찾기
section_content_tag = soup.select('.section-content')
# print(len(section_content_tag)) # 3

# 자식 요소 찾기 (부모 > 자식)
h2_tags = soup.select('.section-content > h2')
# print(h2_tags)

# 후손 선택자 ( 하위 모든 요소를 후손이라 칭함) : (부모 후손)
strong_tag = soup.select('.section-content strong')
print(strong_tag)

# 이전 형제 찾기 : find_previous_sibling()
# 다음 형제 찾기 : find_next_siblink()
em_tag = soup.select_one('.section-content em')
print(em_tag)
print(em_tag.find_previous_sibling())
print(em_tag.find_next_sibling())



[<strong class="highlight">Bold text</strong>]
<em>italic text</em>
<strong class="highlight">Bold text</strong>
<a href="https://www.example.com" target="_blank">links</a>


## 네이버 뉴스 검색 결과에서 제목만 스크래핑


In [47]:
import requests
from bs4 import BeautifulSoup as bs

query = '인공지능' #검색어
url = f'https://search.naver.com/search.naver?ssc=tab.news.all&where=news&sm=tab_jum&query={query}'

try :
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()

except requests.exceptions.Timeout as err:
    print('시간 초과')

except requests.exceptions.RequestException as err:
    print('에러 발생', err)

else:
    soup = bs(response.text, 'html.parser')

    # 파싱된 html 코드에서 뉴스 제목 뽑기

    title_tags = soup.select('.sds-comps-text.sds-comps-text-ellipsis.sds-comps-text-ellipsis-1.sds-comps-text-type-headline1')

    titles = [tag.get_text(strip=True) for tag in title_tags]

    for title in titles:
        print('Title :',title, end = '\n\n')


Title : KAIST,인공지능최대 난제 '발열' 잡을 냉각기술 확보

Title : SK바이오팜, 바이오 USA서인공지능(AI) 신약개발 전략 공개

Title : “사번 001, AI 과장입니다”… SKT ‘인공지능동료’ 파격 실험

Title : 인공지능·디지털 교육 거점 3곳으로 확대

Title : 폴리텍IV대학 대전캠퍼스, 전국인공지능드론 경진서 대상·장려상

Title : 신동빈 회장, "AX는 기업 생존위한 절대 과제". 롯데그룹,인공지능전환...

Title : 인공지능시대 뒤처질라…"中 대학 학위과정 30% 넘게 조정"

Title : 인공지능투자 열풍에 반도체 주도...5월 수출물가 11개월째 상승

Title : 울산시,인공지능선박 기술 주도권 확보 시동

Title : 비투엔, 글로벌 기업 ‘오두’와 맞손…인공지능전사적자원관리 시업...



# 이미지 스크래핑

In [49]:
import os
parent_dir = './naver_news_images'
os.makedirs(parent_dir, exist_ok=True) # 폴더 없으면 생성

image_tags = soup.select(".sds-comps-base-layout.sds-comps-full-layout img")

for index, image_tag in enumerate(image_tags):
    src = image_tag.get('src')

    try :
        image_response = requests.get(src, timeout=10)
        image_response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print("오류 발생: ", e)

    # 저장할 경로
    file_path = os.path.join(parent_dir, f'{index}.jpg')
    with open(file_path, 'wb') as f:
        f.write(image_response.content)

print("이미지 저장 완료")

이미지 저장 완료
